# Thesis hedging experiments

Этот ноутбук привязан к финальному воспроизводимому пайплайну диплома. Он либо использует уже собранные артефакты, либо заново запускает `src/rebuild_thesis_artifacts.py`. Все таблицы и графики, на которые ссылается диплом, берутся из `build/results` и `assets/figures`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

In [ ]:
from src.rebuild_thesis_artifacts import main as rebuild_main

results_dir = ROOT / "build" / "results"
fig_dir = ROOT / "assets" / "figures"
results_dir.exists(), fig_dir.exists()

In [ ]:
if not (results_dir / "policy_comparison.csv").exists():
    rebuild_main()
else:
    print("Artifacts already exist; using cached results.")

In [ ]:
exp1 = pd.read_csv(results_dir / "experiment1_model_comparison.csv")
exp2_spread = pd.read_csv(results_dir / "experiment2_spread.csv")
exp2_liq = pd.read_csv(results_dir / "experiment2_liquidity.csv")
exp3 = pd.read_csv(results_dir / "experiment3_hedging_quantiles.csv")
exp4 = pd.read_csv(results_dir / "policy_comparison.csv")

print("Experiment 1")
print(exp1.to_string(index=False))
print("\nExperiment 3")
print(exp3.to_string(index=False))
print("\nExperiment 4")
print(exp4[exp4["cost_bps"] == 1.0].to_string(index=False))

In [ ]:
df_test = pd.read_csv(results_dir / "df_test_models_snapshot.csv")
print("Rows in test snapshot:", len(df_test))
print(df_test[["S", "K", "T_years", "spread_rel", "open_interest"]].describe())

In [ ]:
from IPython.display import Image, display

for name in [
    "model_rmse_iv.png",
    "spread_bucket_rmse.png",
    "liquidity_bucket_rmse.png",
    "hedging_q05_by_cost.png",
    "policy_cost_tradeoff.png",
    "policy_trade_ratio.png",
    "policy_var95.png",
]:
    print(name)
    display(Image(filename=str(fig_dir / name)))

## Интерпретация

- Микроструктурная RF-модель остаётся лучшей по RMSE(IV) во всех режимах рынка.
- Перенос качества прайсинга в качество дельты неполон: различия в hedging P&L между моделями существенно меньше, чем различия в pricing metrics.
- Наибольший прикладной эффект даёт оптимизация политики ребалансировки: adaptive-policy минимизирует RMSE и совокупные издержки одновременно.